In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

In [4]:
class AnalizadorPrerrequisitos:
    def __init__(self):
        self.df_prereq = None
        self.df_historial = None
        self.resultado = None
    
    def cargar_documentos(self):
        """Solicita y carga los documentos de Excel"""
        print("=== ANALIZADOR DE PRERREQUISITOS ACADÉMICOS ===\n")
        
        # Solicitar archivo de prerrequisitos
        while True:
            try:
                ruta_prereq = "para trabajar pre req progv22025-09-15095342_procesado.xlsx" #input("Ingrese la ruta del archivo 'pre_requisitos.xlsx': ").strip()
                self.df_prereq = pd.read_excel(ruta_prereq)
                print(f"✓ Archivo de prerrequisitos cargado: {len(self.df_prereq)} registros")
                break
            except Exception as e:
                print(f"Error al cargar prerrequisitos: {e}")
                print("Intente nuevamente.\n")
        
        # Solicitar archivo de historial
        while True:
            try:
                ruta_historial = "detalle matricula sistemas pedacito.xlsx" #input("Ingrese la ruta del archivo 'historial_asignaturas.xlsx': ").strip()
                self.df_historial = pd.read_excel(ruta_historial)
                print(f"✓ Archivo de historial cargado: {len(self.df_historial)} registros")
                break
            except Exception as e:
                print(f"Error al cargar historial: {e}")
                print("Intente nuevamente.\n")
    
    def validar_columnas(self):
        """Valida que existan las columnas necesarias"""
        # Columnas requeridas en prerrequisitos
        cols_prereq_req = ['Smbarul_Key_Rule', 'Programa']
        cols_prereq_faltantes = [col for col in cols_prereq_req if col not in self.df_prereq.columns]
        
        if cols_prereq_faltantes:
            raise ValueError(f"Faltan columnas en prerrequisitos: {cols_prereq_faltantes}")
        
        # Columnas requeridas en historial
        cols_hist_req = ['Codigo_Programa', 'Cod materia curso', 'Pidm', 'Materia_Aprobada', 'Calificacion_Final']
        cols_hist_faltantes = [col for col in cols_hist_req if col not in self.df_historial.columns]
        
        if cols_hist_faltantes:
            raise ValueError(f"Faltan columnas en historial: {cols_hist_faltantes}")
        
        print("✓ Todas las columnas necesarias están presentes")
    
    def obtener_columnas_opciones(self):
        """Obtiene las columnas de opciones de prerrequisitos (Opcion_Prereq_1 a Opcion_Prereq_21)"""
        patron = r'^Opcion_Prereq_\d+$'
        columnas_opciones = [col for col in self.df_prereq.columns if re.match(patron, col)]
        columnas_opciones.sort(key=lambda x: int(x.split('_')[-1]))  # Ordenar numéricamente
        return columnas_opciones
    
    def parsear_prerrequisito(self, prereq_str):
        """
        Parsea una cadena de prerrequisito y devuelve una lista de códigos de asignaturas
        Ejemplos:
        - "IGL4990" -> ["IGL4990"]
        - "DIG0020 & HUM4040" -> ["DIG0020", "HUM4040"]
        """
        if pd.isna(prereq_str) or str(prereq_str).strip() == '':
            return []
        
        prereq_str = str(prereq_str).strip()
        
        # Si contiene '&', son múltiples prerrequisitos
        if '&' in prereq_str:
            codigos = [codigo.strip() for codigo in prereq_str.split('&')]
            return [codigo for codigo in codigos if codigo]  # Filtrar vacíos
        else:
            return [prereq_str] if prereq_str else []
    
    def verificar_prerrequisitos_estudiante(self, pidm, codigos_prereq):
        """
        Verifica si un estudiante ha aprobado todos los prerrequisitos de una lista
        Retorna: (cumple: bool, detalles: list)
        """
        if not codigos_prereq:
            return True, []
        
        historial_estudiante = self.df_historial[self.df_historial['Pidm'] == pidm]
        detalles = []
        
        for codigo in codigos_prereq:
            materia_registro = historial_estudiante[
                historial_estudiante['Cod materia curso'] == codigo
            ]
            
            if materia_registro.empty:
                detalles.append({
                    'codigo': codigo,
                    'encontrada': False,
                    'aprobada': False,
                    'nota': None
                })
            else:
                # Tomar el último registro de la materia
                ultimo_registro = materia_registro.iloc[-1]
                aprobada = str(ultimo_registro['Materia_Aprobada']).upper() == 'Y'
                nota = ultimo_registro['Calificacion_Final']
                
                detalles.append({
                    'codigo': codigo,
                    'encontrada': True,
                    'aprobada': aprobada,
                    'nota': nota
                })
        
        # Todos deben estar aprobados para que se cumpla la opción
        cumple = all(detalle['aprobada'] for detalle in detalles)
        return cumple, detalles
    
    def procesar_prerrequisitos(self):
        """Procesa todos los registros y determina los prerrequisitos cumplidos"""
        print("\n=== PROCESANDO PRERREQUISITOS ===")
        
        self.validar_columnas()
        columnas_opciones = self.obtener_columnas_opciones()
        
        print(f"Encontradas {len(columnas_opciones)} columnas de opciones de prerrequisitos")
        print("Procesando registros...")
        
        # Crear DataFrame resultado basado en el historial
        self.resultado = self.df_historial.copy()
        
        # Agregar columnas para resultados
        self.resultado['Prerrequisito_Estado'] = ''
        self.resultado['Prerrequisitos_Encontrados'] = ''
        self.resultado['Notas_Prerrequisitos'] = ''
        
        total_registros = len(self.resultado)
        
        for idx, row in self.resultado.iterrows():
            if idx % 1000 == 0:
                print(f"Procesando registro {idx + 1}/{total_registros}")
            
            pidm = row['Pidm']
            cod_materia = row['Cod materia curso']
            
            # Buscar prerrequisitos para esta materia
            prereq_info = self.df_prereq[
                self.df_prereq['Smbarul_Key_Rule'] == cod_materia
            ]
            
            if prereq_info.empty:
                # No hay información de prerrequisitos para esta materia
                self.resultado.at[idx, 'Prerrequisito_Estado'] = 'No tiene prerrequisito'
                continue
            
            # Hay información de prerrequisitos, buscar primera opción que cumple
            prereq_encontrado = False
            prereq_info_row = prereq_info.iloc[0]  # Tomar el primer registro
            
            for col_opcion in columnas_opciones:
                opcion_value = prereq_info_row[col_opcion]
                
                if pd.isna(opcion_value) or str(opcion_value).strip() == '':
                    continue
                
                # Parsear prerrequisitos de esta opción
                codigos_prereq = self.parsear_prerrequisito(opcion_value)
                
                if not codigos_prereq:
                    continue
                
                # Verificar si el estudiante cumple esta opción
                cumple, detalles = self.verificar_prerrequisitos_estudiante(pidm, codigos_prereq)
                
                if cumple:
                    # Estudiante cumple esta opción de prerrequisito
                    codigos_str = ', '.join([d['codigo'] for d in detalles])
                    notas_str = ', '.join([str(d['nota']) if d['nota'] is not None else 'N/A' 
                                         for d in detalles])
                    
                    self.resultado.at[idx, 'Prerrequisito_Estado'] = 'Prerrequisito cumplido'
                    self.resultado.at[idx, 'Prerrequisitos_Encontrados'] = codigos_str
                    self.resultado.at[idx, 'Notas_Prerrequisitos'] = notas_str
                    prereq_encontrado = True
                    break
            
            if not prereq_encontrado:
                # Tiene prerrequisitos pero no se encontraron cumplidos en el historial
                self.resultado.at[idx, 'Prerrequisito_Estado'] = 'Tiene prerrequisito y no se encontró en la historia'
        
        print(f"✓ Procesamiento completado: {total_registros} registros analizados")
    
    def generar_reporte(self):
        """Genera un reporte resumen del análisis"""
        if self.resultado is None:
            print("No hay resultados para reportar")
            return
        
        print("\n=== REPORTE DE RESULTADOS ===")
        
        resumen = self.resultado['Prerrequisito_Estado'].value_counts()
        print("\nResumen por estado:")
        for estado, cantidad in resumen.items():
            porcentaje = (cantidad / len(self.resultado)) * 100
            print(f"- {estado}: {cantidad} ({porcentaje:.1f}%)")
        
        print(f"\nTotal de registros analizados: {len(self.resultado)}")
    
    def guardar_resultados(self):
        """Guarda los resultados en un archivo Excel"""
        if self.resultado is None:
            print("No hay resultados para guardar")
            return
        
        timestamp = pd.Timestamp.now().strftime("%Y-%m-%d%H%M%S")
        nombre_archivo = "resultados_prerrequisitos_"+timestamp+".xlsx"

        
        try:
            self.resultado.to_excel(nombre_archivo, index=False)
            print(f"✓ Resultados guardados en: {nombre_archivo}")
        except Exception as e:
            print(f"Error al guardar resultados: {e}")
    
    def ejecutar(self):
        """Ejecuta el análisis completo"""
        try:
            self.cargar_documentos()
            self.procesar_prerrequisitos()
            self.generar_reporte()
            self.guardar_resultados()
            print("\n¡Análisis completado exitosamente!")
        except Exception as e:
            print(f"\nError durante la ejecución: {e}")
            print("Verifique los archivos y vuelva a intentar.")

# Función principal
def main():
    analizador = AnalizadorPrerrequisitos()
    analizador.ejecutar()

if __name__ == "__main__":
    main()

=== ANALIZADOR DE PRERREQUISITOS ACADÉMICOS ===

✓ Archivo de prerrequisitos cargado: 3101 registros
✓ Archivo de historial cargado: 28512 registros

=== PROCESANDO PRERREQUISITOS ===
✓ Todas las columnas necesarias están presentes
Encontradas 21 columnas de opciones de prerrequisitos
Procesando registros...
Procesando registro 1/28512
Procesando registro 1001/28512
Procesando registro 2001/28512
Procesando registro 3001/28512
Procesando registro 4001/28512
Procesando registro 5001/28512
Procesando registro 6001/28512
Procesando registro 7001/28512
Procesando registro 8001/28512
Procesando registro 9001/28512
Procesando registro 10001/28512
Procesando registro 11001/28512
Procesando registro 12001/28512
Procesando registro 13001/28512
Procesando registro 14001/28512
Procesando registro 15001/28512
Procesando registro 16001/28512
Procesando registro 17001/28512
Procesando registro 18001/28512
Procesando registro 19001/28512
Procesando registro 20001/28512
Procesando registro 21001/28512